In [0]:
dbutils.widgets.text("input_bcn2acn_parquet_file", "/Volumes/7_outgoing/pharos/accession_numbers/pharos_20260828_bcn_pid_acn.parquet", "parquet file generated by BCN to AccessionNbr notebook")
dbutils.widgets.text("input_extracted_dcm_tags_file", "/Volumes/7_outgoing/pharos/imaging/pharos_20260828_metadata.parquet", "parquet file generated by ExtractedMetadata notebook")
dbutils.widgets.text("input_llm_verifier_result_file_prefix", "/Volumes/1_inland/sectra/vone/20260626_131626_llm_verifier2.parquet_batch", "parquet files generated by LLM PII verifier")
dbutils.widgets.text("output_parquet_file", "/Volumes/1_inland/sectra/vone/20260626_131626_llm_verifier2.parquet_batch", "Output parquet file path")

In [0]:
input_bcn2acn_parquet_file = dbutils.widgets.get("input_bcn2acn_parquet_file")
bcn2acn_df = spark.read.parquet(input_bcn2acn_parquet_file)
display(bcn2acn_df.limit(1000))
print(bcn2acn_df.count())

In [0]:
input_extracted_dcm_tags_file = dbutils.widgets.get("input_extracted_dcm_tags_file")
extracted_dcm_tags_df = spark.read.parquet(input_extracted_dcm_tags_file)
extracted_dcm_tags_df = extracted_dcm_tags_df.withColumnRenamed("accession_number", "exported_accession_number")
extracted_dcm_tags_df = extracted_dcm_tags_df.drop("scan_dir_path").drop("dcm_file")
display(extracted_dcm_tags_df.limit(1000))
print(extracted_dcm_tags_df.count())
print(extracted_dcm_tags_df.select("exported_accession_number").distinct().count())

In [0]:
joined_df = bcn2acn_df.join(extracted_dcm_tags_df, bcn2acn_df.requested_accession_number == extracted_dcm_tags_df.exported_accession_number, "left")
display(joined_df.orderBy("bcn_id"))
print(joined_df.count())

In [0]:
from pyspark.sql.functions import regexp_replace
from pyspark.sql.functions import from_json, schema_of_json
from pyspark.sql.functions import lit
from pyspark.sql.functions import col
from pyspark.sql.functions import regexp_extract
from pyspark.sql import functions as F

llm_file_prefix = dbutils.widgets.get("input_llm_verifier_result_file_prefix")

llm_df = spark.read.parquet(f"{llm_file_prefix}*.parquet")
#llm_df = llm_df.withColumn(
#    "path",
#    regexp_replace(
#        regexp_replace("path", "/Volumes/8_dev/pacs/anon_images/20260626_131626", ""),
#        "_anon",
#        ""
#    )
#)

llm_df = llm_df.withColumn(
    "path",
    F.concat_ws(
        "/",
        F.element_at(F.split("path", "/"), -3),
        F.element_at(F.split("path", "/"), -2),
        F.element_at(F.split("path", "/"), -1),
    )
)


#llm_df = llm_df.withColumn(
#    "subdir",
#    regexp_extract("path", r"(/?\d{6}/\d{6}/)", 1)
#)

llm_df = llm_df.withColumn(
        "subdir",
        F.concat(
            F.element_at(F.split("path", "/"), -3),
            F.lit("/"),
            F.element_at(F.split("path", "/"), -2),
            F.lit("/")
        )
    )

sample_json = llm_df.select("llm_result").filter(col("llm_result").isNotNull()).limit(1).collect()[0]["llm_result"]
json_schema = schema_of_json(sample_json)
llm_df = llm_df.withColumn(
    "llm_result_json", from_json(col("llm_result"), json_schema)
).withColumn(
    "is_text_detected", col("llm_result_json.is_text_detected")
).withColumn(
    "is_personal_information_detected", col("llm_result_json.is_personal_information_detected")
)
llm_df = llm_df.select("subdir", "path", "is_personal_information_detected").filter(col("is_personal_information_detected") == "yes")
#llm_df = llm_df.join(joined_df.select("accession_number", "path"), on="path", how="left")


In [0]:
display(llm_df.filter("is_personal_information_detected == 'yes'").limit(100))

In [0]:
from pyspark.sql.functions import collect_set, col

agg_df = (
    llm_df.filter(col("is_personal_information_detected") == "yes")
    .groupBy("subdir")
    .agg(collect_set("path").alias("pii_detected_by_llm"))
)
display(agg_df)

In [0]:
joined_llm_df = joined_df.join(agg_df, joined_df.scan_dir == agg_df.subdir, how="left").drop("subdir")
display(joined_llm_df.limit(1000))


In [0]:

joined_llm_df.coalesce(1).write.mode("overwrite").parquet(dbutils.widgets.get("output_parquet_file"))